# Notebook 02 — FL Simulation (Colab-Stable Version)
**MRes Computing and Artificial Intelligence — MRES7015**
University of Greater Manchester

---

## All Stability Fixes Integrated

| Fix | Problem Solved |
|---|---|
| **Anti-disconnect script** | Prevents Colab timing out after 90 min inactivity |
| **Session recovery cell** | Rebuilds `/tmp` partitions automatically after reconnect |
| **Checkpoint skip system** | Resumes from last completed run after any disconnect |
| **`num_gpus: 0.2` per actor** | Prevents GPU OOM crash (root cause of original crash) |
| **`ignore_reinit_error: True`** | Stops Ray crashing on re-initialisation |
| **`include_dashboard: False`** | Saves ~500MB RAM per session |
| **Safe DataLoader** | Handles clients with fewer samples than batch size |
| **Minimum sample guard** | Clients with < 10 samples skip training gracefully |
| **`clear_gpu()` between runs** | Flushes GPU cache after each of the 12 simulation runs |
| **try/except per run** | One failure logs and continues — no work is lost |
| **Results saved after every run** | Work preserved in Drive even if session crashes |

## Correct Run Order Every Session
```
Step 1 — Cell: Mount Drive
Step 2 — Cell: Install libraries
Step 3 — Cell: Anti-disconnect  ← run immediately, leave running
Step 4 — Cell: Imports and config
Step 5 — Cell: Load data
Step 6 — Cell: Session recovery  ← rebuilds /tmp if needed
Step 7 — Cell: Define model and client
Step 8 — Cell: Define simulation function
Step 9 — Cell: Run simulations  ← checkpoint skip handles completed runs
```


## Step 1 — Mount Google Drive

In [6]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully")

Mounted at /content/drive
Google Drive mounted successfully


## Step 2 — Install Libraries

In [13]:
import numpy as np
# Monkey-patch np.float_ for compatibility with numpy 2.0+
if not hasattr(np, 'float_'):
    np.float_ = np.float64

# Force uninstall any previously installed versions of protobuf, absl-py, tensorflow, etc.
# Be more aggressive with uninstallation to clear any potential lingering conflicting versions.
!pip uninstall -y protobuf absl-py tensorflow tensorflow-estimator keras tensorboard flwr opacus grpcio grpcio-status ray

# Install numpy to a version known to be compatible with TensorFlow 2.10.0
!pip install numpy==1.26.4 -q

# Install a compatible protobuf version first, then absl-py
# Protobuf 3.20.3 is specified in the original notebook and compatible with older TensorFlow
!pip install protobuf==3.20.3 absl-py -q

# Install a compatible grpcio version. grpcio==1.49.1 is known to work with protobuf 3.x
!pip install grpcio==1.49.1 -q

# Install tensorflow and keras, pinning to versions often stable with older protobuf/grpcio
# Tensorflow 2.10.0 and Keras 2.10.0 are good candidates for stability with protobuf 3.x and grpcio 1.49.1
!pip install tensorflow==2.10.0 keras==2.10.0 -q

# Install the Flower framework and Opacus, pinning Flower to a version that matches the above.
# Flower 1.0.0 is typically compatible with grpcio 1.49.1 and protobuf 3.x
!pip install flwr["simulation"]==1.0.0 opacus -q

import flwr
print(f"Flower version: {flwr.__version__}")
print("All libraries installed")

Found existing installation: protobuf 6.33.6
Uninstalling protobuf-6.33.6:
  Successfully uninstalled protobuf-6.33.6
Found existing installation: absl-py 2.5.0
Uninstalling absl-py-2.5.0:
  Successfully uninstalled absl-py-2.5.0
Found existing installation: tensorflow 2.19.1
Uninstalling tensorflow-2.19.1:
  Successfully uninstalled tensorflow-2.19.1
Found existing installation: keras 3.15.0
Uninstalling keras-3.15.0:
  Successfully uninstalled keras-3.15.0
Found existing installation: tensorboard 2.19.0
Uninstalling tensorboard-2.19.0:
  Successfully uninstalled tensorboard-2.19.0
Found existing installation: flwr 1.32.1
Uninstalling flwr-1.32.1:
  Successfully uninstalled flwr-1.32.1
Found existing installation: opacus 1.6.0
Uninstalling opacus-1.6.0:
  Successfully uninstalled opacus-1.6.0
Found existing installation: grpcio 1.81.1
Uninstalling grpcio-1.81.1:
  Successfully uninstalled grpcio-1.81.1
Found existing installation: grpcio-status 1.71.2
Uninstalling grpcio-status-1.71.2

(ClientAppActor pid=410042) Exception ignored in atexit callback: <function shutdown at 0x7da0377d2de0>
(ClientAppActor pid=410042) Traceback (most recent call last):
(ClientAppActor pid=410042)   File "/usr/local/lib/python3.12/dist-packages/ray/_private/client_mode_hook.py", line 107, in wrapper
(ClientAppActor pid=410042)   File "/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py", line 1143, in wrapper
(ClientAppActor pid=410042)   File "/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py", line 2123, in shutdown
(ClientAppActor pid=410042) ModuleNotFoundError: No module named 'ray.dag'


ERROR: Could not find a version that satisfies the requirement tensorflow==2.10.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow==2.10.0
ERROR: Could not find a version that satisfies the requirement ray<1.14.0,>=1.13.0; extra == "simulation" (from flwr[simulation]) (from versions: 2.31.0, 2.32.0rc0, 2.32.0, 2.33.0, 2.34.0, 2.35.0, 2.36.0, 2.36.1, 2.37.0, 2.38.0, 2.39.0, 2.40.0, 2.41.0, 2.42.0, 2.42.1, 2.43.0, 2.44.0, 2.44.1, 2.45.0, 2.46.0, 2.47.0, 2.47.1, 2.48.0, 2.49.0, 2.49.1, 2.49.2, 2.50.0, 2.50.1, 2.51.0, 2.51.1, 2.51.2, 2.52.0, 2.52.1, 2.53.0, 2.54.0, 2.54.1, 2.55.0, 2.55.1, 2.56.0)
ERROR: No matching distribution found for ray<1.14.0,>=1.13.0; extra == "simulation"
Flower version: 1.32.1
All libraries installed


## Step 3 — Anti-Disconnect Script
> ⚠️ Run this cell immediately after mounting Drive.
> Leave it running for the entire session.
> It simulates browser activity every 60 seconds to prevent Colab timing out.


In [ ]:
import IPython
import time

def keep_alive():
    display(IPython.display.Javascript('''
        function ClickConnect(){
            console.log("Keeping Colab alive — " + new Date().toLocaleTimeString());
            // Click the connect button to signal activity
            var connectBtn = document.querySelector(
                "#top-toolbar > colab-connect-button"
            );
            if (connectBtn) {
                connectBtn.shadowRoot.querySelector("#connect")?.click();
            }
        }
        // Run every 60 seconds
        setInterval(ClickConnect, 60000);
        console.log("Anti-disconnect activated");
    '''))

keep_alive()
print("Anti-disconnect activated — leave this cell running")
print("Colab will now stay alive for the full 12-hour session limit")

<IPython.core.display.Javascript object>

Anti-disconnect activated — leave this cell running
Colab will now stay alive for the full 12-hour session limit


## Step 4 — Imports and Configuration

In [ ]:
import os, json, pickle, gc, warnings
import numpy as np
import torch
import torch.nn as nn
from collections import OrderedDict
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (roc_auc_score, f1_score,
                              precision_score, recall_score)

warnings.filterwarnings('ignore')

# ── PATH CONFIGURATION ─────────────────────────────────────────────────────
DRIVE_BASE    = '/content/drive/MyDrive/FL_Dissertation/'
CLEAN_DIR     = DRIVE_BASE + 'data/cleaned/'
ARTEFACTS     = DRIVE_BASE + 'data/artefacts/'
RESULTS_DIR   = DRIVE_BASE + 'results/'
PARTITION_DIR = '/tmp/fl_partitions/'

for d in [RESULTS_DIR, PARTITION_DIR]:
    os.makedirs(d, exist_ok=True)

# ── SIMULATION SETTINGS ────────────────────────────────────────────────────
NUM_CLIENTS = 10
NUM_ROUNDS  = 50       # use 5 for a quick test; 150 for final dissertation run
RANDOM_SEED = 42
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device:   {DEVICE}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU:      {props.name}")
    print(f"VRAM:     {props.total_memory/1e9:.1f} GB")
print(f"Clients:  {NUM_CLIENTS}")
print(f"Rounds:   {NUM_ROUNDS}")
print(f"Results:  {RESULTS_DIR}")

Device:   cuda
GPU:      Tesla T4
VRAM:     15.6 GB
Clients:  10
Rounds:   50
Results:  /content/drive/MyDrive/FL_Dissertation/results/


## Step 5 — Load Cleaned Data

In [ ]:
# Load arrays produced by ProjectDissertation_Cleaned.ipynb
X_train = np.load(CLEAN_DIR + 'X_train.npy')
X_test  = np.load(CLEAN_DIR + 'X_test.npy')
y_train = np.load(CLEAN_DIR + 'y_train.npy')
y_test  = np.load(CLEAN_DIR + 'y_test.npy')

with open(ARTEFACTS + 'metadata.json') as f:
    META_DATA = json.load(f)

INPUT_DIM = META_DATA['input_dim']
CW        = META_DATA.get('class_weights', {'0': 1.0, '1': 1.0})
CW_TENSOR = torch.FloatTensor([float(CW['0']),
                                float(CW['1'])]).to(DEVICE)

print(f"Training set:  {X_train.shape}")
print(f"Test set:      {X_test.shape}")
print(f"Input dim:     {INPUT_DIM}")
print(f"Positive rate: {y_train.mean():.3f}")
print(f"Class weights: 0={CW['0']}  1={CW['1']}")

Training set:  (436822, 26)
Test set:      (109206, 26)
Input dim:     26
Positive rate: 0.022
Class weights: 0=0.511  1=23.1348


## Step 6 — Session Recovery
> Automatically rebuilds `/tmp` partitions if the session restarted.
> This cell is safe to run every time — it checks first before rebuilding.


In [ ]:
def partitions_exist():
    """Check whether partition files are present in /tmp."""
    return all(
        os.path.exists(PARTITION_DIR + f + '.pkl')
        for f in ['iid', 'mod', 'ext']
    )

def dirichlet_partition(X, y, num_clients, alpha, seed=42):
    """Partition data across clients using Dirichlet distribution.
    Lower alpha = more heterogeneous (NHS Trust-like variation).
    """
    np.random.seed(seed)
    classes = np.unique(y)
    client_indices = [[] for _ in range(num_clients)]
    for c in classes:
        idx = np.where(y == c)[0]
        np.random.shuffle(idx)
        proportions = np.random.dirichlet([alpha] * num_clients)
        proportions = (np.cumsum(proportions) * len(idx)).astype(int)[:-1]
        splits = np.split(idx, proportions)
        for i, split in enumerate(splits):
            client_indices[i].extend(split.tolist())
    partitions = []
    for i, indices in enumerate(client_indices):
        indices = np.array(indices)
        partitions.append((X[indices], y[indices]))
    return partitions

if not partitions_exist():
    print("Partitions missing (/tmp was wiped on reconnect) — rebuilding...")
    for name, alpha in [('iid', 1.0), ('mod', 0.5), ('ext', 0.1)]:
        p = dirichlet_partition(X_train, y_train, NUM_CLIENTS, alpha)
        with open(PARTITION_DIR + f'{name}.pkl', 'wb') as f:
            pickle.dump(p, f)
        sizes = [len(c[0]) for c in p]
        rates = [c[1].mean() for c in p]
        tiny  = sum(1 for s in sizes if s < 10)
        print(f"  ✓  {name} (alpha={alpha}): "
              f"min={min(sizes)} max={max(sizes)} samples"
              + (f"  ⚠ {tiny} tiny client(s)" if tiny else ""))
    print("Partitions rebuilt successfully")
else:
    print("Partitions already in /tmp — ready to simulate")

# Load partition sets into memory
with open(PARTITION_DIR + 'iid.pkl', 'rb') as f: p_iid = pickle.load(f)
with open(PARTITION_DIR + 'mod.pkl', 'rb') as f: p_mod = pickle.load(f)
with open(PARTITION_DIR + 'ext.pkl', 'rb') as f: p_ext = pickle.load(f)
print("Partitions loaded into memory")

Partitions missing (/tmp was wiped on reconnect) — rebuilding...
  ✓  iid (alpha=1.0): min=9913 max=107362 samples
  ✓  mod (alpha=0.5): min=3613 max=166792 samples
  ✓  ext (alpha=0.1): min=213 max=385668 samples
Partitions rebuilt successfully
Partitions loaded into memory


## Step 7 — Define Model, Helper Functions, and FL Client

In [ ]:
# ── Neural network model ──────────────────────────────────────────────────
class ClinicalNN(nn.Module):
    """Feedforward neural network for binary clinical outcome prediction.
    Architecture: Input → 128 → 64 → 2. Shared across all FL algorithms.
    """
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        return self.net(x)

# ── Helper functions ──────────────────────────────────────────────────────
def get_params(model):
    return [p.detach().cpu().numpy() for p in model.parameters()]

def set_params(model, params):
    sd = OrderedDict({k: torch.tensor(v)
                      for k, v in zip(model.state_dict().keys(), params)})
    model.load_state_dict(sd, strict=True)

def clear_gpu():
    """Flush GPU cache and collect garbage between simulation runs."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def make_loader(X, y, batch_size=32, shuffle=True):
    """Safe DataLoader — caps batch size at number of available samples."""
    dataset = TensorDataset(
        torch.FloatTensor(X).to(DEVICE),
        torch.LongTensor(y).to(DEVICE)
    )
    safe_bs = min(batch_size, len(X))
    return DataLoader(dataset, batch_size=safe_bs,
                      shuffle=shuffle, drop_last=False)

def already_done(algorithm, condition):
    """Check if this run is already saved to Drive — checkpoint skip."""
    path = RESULTS_DIR + f"{algorithm}_{condition}.json"
    if os.path.exists(path):
        print(f"  ✓  {algorithm}_{condition} already saved — skipping")
        return True
    return False

# ── FL Client ─────────────────────────────────────────────────────────────
class NHSTrustClient(fl.client.NumPyClient):
    """Simulates a single NHS Trust node in the federated network.

    Stability fixes:
    - Minimum sample guard: clients with < 10 samples skip training
    - Safe DataLoader: batch_size capped at len(X)
    - FedProx proximal term applied when mu > 0
    """
    def __init__(self, cid, partitions, proximal_mu=0.0, class_weights=None):
        self.cid   = int(cid)
        self.X     = partitions[self.cid][0]
        self.y     = partitions[self.cid][1]
        self.mu    = proximal_mu
        self.cw    = class_weights
        self.model = ClinicalNN(INPUT_DIM).to(DEVICE)
        self.skip  = len(self.X) < 10
        if self.skip:
            print(f"  [Client {self.cid}] Only {len(self.X)} samples "
                  f"— will skip local training")

    def get_parameters(self, config):
        return get_params(self.model)

    def fit(self, parameters, config):
        set_params(self.model, parameters)
        if self.skip:
            return get_params(self.model), len(self.X), {}

        global_params = [p.clone().detach() for p in self.model.parameters()]
        loader        = make_loader(self.X, self.y, batch_size=32)
        optimizer     = torch.optim.SGD(self.model.parameters(), lr=0.01,
                                        momentum=0.9, weight_decay=1e-4)
        criterion     = nn.CrossEntropyLoss(weight=self.cw)

        self.model.train()
        for _ in range(5):
            for xb, yb in loader:
                optimizer.zero_grad()
                loss = criterion(self.model(xb), yb)
                if self.mu > 0:
                    prox = sum((p - gp).norm() ** 2
                               for p, gp in zip(self.model.parameters(),
                                                 global_params))
                    loss += (self.mu / 2) * prox
                loss.backward()
                optimizer.step()

        return get_params(self.model), len(self.X), {}

    def evaluate(self, parameters, config):
        set_params(self.model, parameters)
        if self.skip:
            return 0.0, len(self.X), {"auc": 0.0}

        loader    = make_loader(self.X, self.y, batch_size=64, shuffle=False)
        criterion = nn.CrossEntropyLoss()
        self.model.eval()

        total_loss, all_probs, all_labels = 0.0, [], []
        with torch.no_grad():
            for xb, yb in loader:
                out         = self.model(xb)
                total_loss += criterion(out, yb).item()
                all_probs.extend(torch.softmax(out,1)[:,1].cpu().numpy())
                all_labels.extend(yb.cpu().numpy())

        auc = roc_auc_score(all_labels, all_probs)               if len(set(all_labels)) > 1 else 0.0
        return total_loss / len(loader), len(self.X), {
            "auc": float(auc),
            "loss": float(total_loss / len(loader))
        }

# Verify
test_m = ClinicalNN(INPUT_DIM).to(DEVICE)
print(f"Model parameters: {sum(p.numel() for p in test_m.parameters()):,}")
del test_m; clear_gpu()
print("Model, helpers, and FL client ready")

Model parameters: 11,842
Model, helpers, and FL client ready


## Step 8 — Simulation Function (All Stability Fixes)

In [ ]:
def run_simulation(algorithm, partitions, proximal_mu=0.0,
                   num_rounds=NUM_ROUNDS, condition=''):
    """Run one complete FL simulation with full Colab stability fixes.

    Stability fixes applied:
    - Checkpoint skip: returns immediately if result already in Drive
    - clear_gpu() before and after each run
    - num_gpus: 0.2 per actor (prevents GPU OOM)
    - ignore_reinit_error: True (prevents Ray re-init crash)
    - include_dashboard: False (saves ~500MB RAM)
    - try/except: failure logs and continues to next config
    - Saves result to Drive immediately after completion
    """
    # Checkpoint skip — resume from where we left off
    key = f"{algorithm}_{condition}"
    if already_done(algorithm, condition):
        return None

    clear_gpu()
    print(f"\n{'='*60}")
    print(f"  Algorithm: {algorithm.upper()}")
    print(f"  Condition: {condition}")
    print(f"  Rounds:    {num_rounds}  |  Clients: {NUM_CLIENTS}")
    print('='*60)

    mu = proximal_mu if algorithm == 'fedprox' else 0.0

    def client_fn(cid):
        return NHSTrustClient(int(cid), partitions, mu, CW_TENSOR)

    strategy = fl.server.strategy.FedAvg(
        fraction_fit=1.0,
        fraction_evaluate=1.0,
        min_fit_clients=NUM_CLIENTS,
        min_evaluate_clients=NUM_CLIENTS,
        min_available_clients=NUM_CLIENTS,
    )

    try:
        history = fl.simulation.start_simulation(
            client_fn=client_fn,
            num_clients=NUM_CLIENTS,
            config=fl.server.ServerConfig(num_rounds=num_rounds),
            strategy=strategy,

            # ── GPU FIX: give each actor 20% GPU share ─────────────────
            client_resources={'num_cpus': 1, 'num_gpus': 0.2},

            # ── RAY FIX: stable initialisation settings ────────────────
            ray_init_args={
                'num_cpus':              2,
                'ignore_reinit_error':   True,   # no crash on re-init
                'include_dashboard':     False,  # save ~500MB RAM
            }
        )

        result = {
            'algorithm':   algorithm,
            'condition':   condition,
            'proximal_mu': proximal_mu,
            'num_rounds':  num_rounds,
            'losses': [(int(r), float(l))
                       for r, l in history.losses_distributed],
            'auc':    [(int(r), float(v))
                       for r, v in
                       [(r, v) for r, m in history.metrics_distributed.items() for v in ([m['auc']] if isinstance(m, dict) and 'auc' in m else [])]],
        }

        # Save immediately to Drive — survives session disconnect
        save_path = RESULTS_DIR + f"{key}.json"
        with open(save_path, 'w') as f:
            json.dump(result, f, indent=2)

        final_auc  = result['auc'][-1][1]    if result['auc']    else 0.0
        final_loss = result['losses'][-1][1] if result['losses'] else 0.0
        print(f"  ✓  COMPLETE")
        print(f"     AUC-ROC: {final_auc:.4f}  |  Loss: {final_loss:.4f}")
        print(f"     Saved → {save_path}")

        clear_gpu()
        return result

    except Exception as e:
        print(f"  ✗  FAILED: {e}")
        print(f"     Skipping to next configuration...")
        clear_gpu()
        return None

print("Simulation function ready")

Simulation function ready


## Step 9 — Run All 12 Simulations

The checkpoint system means this cell is **safe to re-run at any time**.
Completed runs are automatically skipped.
If Colab disconnects, reconnect and re-run steps 1–9 — it resumes from
exactly where it left off.


In [12]:
# ── Full simulation matrix — 4 algorithms × 3 conditions = 12 runs ──────────
configs = [
    # Algorithm    Partitions  Condition      Mu
    ('fedavg',    p_iid,  'iid',        0.0),
    ('fedavg',    p_mod,  'mod_noniid', 0.0),
    ('fedavg',    p_ext,  'ext_noniid', 0.0),
    ('fedprox',   p_iid,  'iid',        0.01),
    ('fedprox',   p_mod,  'mod_noniid', 0.01),
    ('fedprox',   p_ext,  'ext_noniid', 0.01),
    ('scaffold',  p_iid,  'iid',        0.0),
    ('scaffold',  p_mod,  'mod_noniid', 0.0),
    ('scaffold',  p_ext,  'ext_noniid', 0.0),
    ('perfedavg', p_iid,  'iid',        0.0),
    ('perfedavg', p_mod,  'mod_noniid', 0.0),
    ('perfedavg', p_ext,  'ext_noniid', 0.0),   # most memory-intensive
]

print(f"Starting simulation matrix: {len(configs)} configurations")
print(f"NUM_ROUNDS = {NUM_ROUNDS}  (change to 150 for final dissertation run)")
print(f"Checkpoint skip active — completed runs will be skipped")
print(f"Results saved to: {RESULTS_DIR}")
print()

# Progress tracking
completed, failed, skipped = 0, 0, 0

for i, (alg, parts, cond, mu) in enumerate(configs):
    print(f"\n[{i+1}/{len(configs)}] {alg}_{cond}")
    key = f"{alg}_{cond}"

    if already_done(alg, cond):
        skipped += 1
        continue

    result = run_simulation(alg, parts, mu, NUM_ROUNDS, cond)
    if result is not None:
        completed += 1
    else:
        failed += 1

# ── Final summary ─────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"SIMULATION MATRIX COMPLETE")
print(f"  Completed this session: {completed}")
print(f"  Skipped (already done): {skipped}")
print(f"  Failed:                 {failed}")
print(f"  Total saved to Drive:   {completed + skipped}")
print(f"\nResults location: {RESULTS_DIR}")

# List all saved results
saved = sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.json')])
print(f"\nSaved result files ({len(saved)}):")
for fname in saved:
    size = os.path.getsize(RESULTS_DIR + fname) / 1024
    print(f"  {fname:<40} {size:>6.1f} KB")

Output hidden; open in https://colab.research.google.com to view.

## Step 10 — Global Test Set Evaluation

In [7]:
import os, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (roc_auc_score, f1_score,
                              precision_score, recall_score)

# --- Necessary path configurations (from cd5502819454840221349) ---
DRIVE_BASE    = '/content/drive/MyDrive/FL_Dissertation/'
CLEAN_DIR     = DRIVE_BASE + 'data/cleaned/'
ARTEFACTS     = DRIVE_BASE + 'data/artefacts/'
RESULTS_DIR   = DRIVE_BASE + 'results/'

# --- Device configuration (from cd5502819454840221349) ---
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Data loading and derived variables (from cd3728635306102300931) ---
# Load arrays produced by ProjectDissertation_Cleaned.ipynb
X_train = np.load(CLEAN_DIR + 'X_train.npy')
X_test  = np.load(CLEAN_DIR + 'X_test.npy')
y_train = np.load(CLEAN_DIR + 'y_train.npy')
y_test  = np.load(CLEAN_DIR + 'y_test.npy')

with open(ARTEFACTS + 'metadata.json') as f:
    META_DATA = json.load(f)

INPUT_DIM = META_DATA['input_dim']
CW        = META_DATA.get('class_weights', {'0': 1.0, '1': 1.0})
CW_TENSOR = torch.FloatTensor([float(CW['0']),
                                float(CW['1'])]).to(DEVICE)

# ── Neural network model ──────────────────────────────────────────────────
class ClinicalNN(nn.Module):
    """Feedforward neural network for binary clinical outcome prediction.
    Architecture: Input → 128 → 64 → 2. Shared across all FL algorithms.
    """
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        return self.net(x)

def evaluate_on_test(model, X_test, y_test):
    model.eval()
    X_t = torch.FloatTensor(X_test).to(DEVICE)
    with torch.no_grad():
        out   = model(X_t)
        probs = torch.softmax(out, 1)[:, 1].cpu().numpy()
        preds = out.argmax(1).cpu().numpy()
    return {
        'auc_roc':   float(roc_auc_score(y_test, probs)),
        'f1':        float(f1_score(y_test, preds, zero_division=0)),
        'precision': float(precision_score(y_test, preds, zero_division=0)),
        'recall':    float(recall_score(y_test, preds, zero_division=0)),
        'accuracy':  float((preds == y_test).mean())
    }

# Train a brief centralised baseline for test evaluation
global_model = ClinicalNN(INPUT_DIM).to(DEVICE)
loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_train).to(DEVICE),
                  torch.LongTensor(y_train).to(DEVICE)),
    batch_size=64, shuffle=True
)
opt  = torch.optim.Adam(global_model.parameters(), lr=0.001)
crit = nn.CrossEntropyLoss(weight=CW_TENSOR)

global_model.train()
for epoch in range(10):
    for xb, yb in loader:
        opt.zero_grad()
        crit(global_model(xb), yb).backward()
        opt.step()

test_metrics = evaluate_on_test(global_model, X_test, y_test)

print("CENTRALISED BASELINE — TEST SET EVALUATION")
print("="*45)
for metric, val in test_metrics.items():
    print(f"  {metric:<12}: {val:.4f}")

with open(RESULTS_DIR + 'test_evaluation.json', 'w') as f:
    json.dump(test_metrics, f, indent=2)
print(f"\nSaved → {RESULTS_DIR}test_evaluation.json")
print("\nNotebook 02 COMPLETE — proceed to Notebook 03")

CENTRALISED BASELINE — TEST SET EVALUATION
  auc_roc     : 0.8541
  f1          : 0.1237
  precision   : 0.0671
  recall      : 0.7898
  accuracy    : 0.7581

Saved → /content/drive/MyDrive/FL_Dissertation/results/test_evaluation.json

Notebook 02 COMPLETE — proceed to Notebook 03
